In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy.stats import multivariate_normal
from tqdm import tqdm
import cvxpy as cp
from Newsvendor import Newsvendor
from joblib import Parallel, delayed
import torch
import torch.nn as nn
import normflows as nf
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.neighbors import KernelDensity
from pathlib import Path
from time import perf_counter

In [ ]:
########################## Normalizing flow function ###################################
def train_nf_model(latent_size, best_K, hidden_node, hidden_layer, num_bins, block_size, total_epoch, x, device, base_gmm=None, covariance_type="diag", batch_size=64, lr=1e-3, patience=30, val_split=0.2):
    x_np = x.detach().cpu().numpy()
    if base_gmm is None:
        base_gmm = GaussianMixture(
            n_components=best_K,
            covariance_type=covariance_type,
            n_init=10,
            reg_covar=1e-2,
            random_state=42,
        ).fit(x_np)

    if base_gmm.covariance_type != "diag":
        raise ValueError("normflows GaussianMixture q0 currently expects diagonal scale parameters.")

    means = torch.tensor(base_gmm.means_, dtype=torch.float32, device=device)
    weights = torch.tensor(base_gmm.weights_, dtype=torch.float32, device=device)
    stds = torch.tensor(np.sqrt(base_gmm.covariances_), dtype=torch.float32, device=device)

    flows = [nf.flows.AutoregressiveRationalQuadraticSpline(latent_size, hidden_layer, hidden_node, num_bins=num_bins) for _ in range(block_size)]
    q0 = nf.distributions.GaussianMixture(n_modes=best_K, dim=latent_size, loc=means, scale=stds, weights=weights, trainable=False)
    nfm = nf.NormalizingFlow(q0=q0, flows=flows).to(device)
    optimizer = torch.optim.Adam(nfm.parameters(), lr=lr)

    dataset = TensorDataset(x)
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    loss_hist = []
    val_loss_hist = []

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    for epoch in tqdm(range(total_epoch), desc="Training NF", leave=False):
        nfm.train()
        train_loss_epoch = 0.0
        for batch in train_loader:
            x_batch = batch[0].to(device)
            optimizer.zero_grad()
            loss = nfm.forward_kld(x_batch)
            if not torch.isnan(loss):
                loss.backward()
                optimizer.step()
                train_loss_epoch += loss.item()

        nfm.eval()
        val_loss_epoch = 0.0
        with torch.no_grad():
            for batch in val_loader:
                x_batch = batch[0].to(device)
                loss = nfm.forward_kld(x_batch)
                if not torch.isnan(loss):
                    val_loss_epoch += loss.item()

        loss_hist.append(train_loss_epoch)
        val_loss_hist.append(val_loss_epoch)

        if val_loss_epoch < best_val_loss:
            best_val_loss = val_loss_epoch
            patience_counter = 0
            best_model_state = {k: v.detach().cpu().clone() for k, v in nfm.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_model_state is not None:
        nfm.load_state_dict(best_model_state)

    return nfm, loss_hist, base_gmm

def inverse(nfm, x):
    with torch.no_grad():
        z_np = nfm.inverse(x).cpu().numpy()
    return z_np

def forward(nfm, z):
    with torch.no_grad():
        x = nfm.forward(z).cpu().numpy()
    return x

In [ ]:
######################### GMM function ###################################
def make_eps_grid(a_list, b_list):
    eps_grid = []

    for b in b_list:
        for a in a_list:
            eps_grid.append(a * (10 ** b))

    return eps_grid

def NewsVendor_2_Wass(xi, eps, h, b):
    scale = 100 
    xi = xi.astype(float)/ scale
    eps = eps / scale
    N = xi.shape[0]
    lda = cp.Variable(nonneg = True)
    s = cp.Variable(N)
    theta = cp.Variable(nonneg = True)
    q = cp.Variable(nonneg = True)

    const = []
    for i in range(N):
        const.append(cp.norm2(cp.hstack([2 * lda * xi[i] + theta - h, lda * (xi[i]**2) - h * q + s[i] - lda])) <= lda * (xi[i]**2) - h * q + s[i] +lda)
        const.append(cp.norm2(cp.hstack([2 * lda * xi[i] + theta + b, lda * (xi[i]**2) + b * q + s[i] - lda])) <= lda * (xi[i]**2) + b * q + s[i] +lda)
        const.append(lda * (xi[i]**2) - h * q + s[i] >= 0)
        const.append(lda * (xi[i]**2) + b * q + s[i] >= 0)

    obj = cp.Minimize(lda * (eps**2) + (1 / N) * cp.sum(s))
    prob = cp.Problem(obj, const)
    prob.solve(solver = cp.MOSEK, verbose = False)

    return q.value * scale 

def generate_data(n, dim_s, dim_xi, W1, W2, seed=None):
    rng = np.random.default_rng(seed)  
    s = rng.uniform(-2, 2, size=(n, dim_s))
    eps1 = rng.uniform(-2, 2, size=(n, dim_xi))
    eps2 = rng.uniform(-2, 2, size=(n, dim_xi))
    lin  = s @ W1.T
    quad = (s**2) @ W2.T
    xi1 = lin + 50 + eps1
    xi2 = quad + 40 + eps2
    score = s[:, 0]
    prob = 0.5 * (1 + np.tanh(score)) 
    u = rng.random(n)
    xi = np.where(u[:, None] < prob[:, None], xi1, xi2)
    return s, xi

def fit_gmm_by_aic(X, max_K, covariance_type, reg_covar=1e-2, n_init=10, random_state=42):
    k_list = range(1, max_K + 1)
    best_gmm = None
    best_k = None
    best_aic = np.inf
    records = []

    for k in k_list:
        gmm = GaussianMixture(n_components=k, covariance_type=covariance_type, reg_covar=reg_covar, n_init=n_init, random_state=random_state,).fit(X)
        aic = gmm.aic(X)
        bic = gmm.bic(X)
        records.append((k, aic, bic))
        if aic < best_aic:
            best_aic = aic
            best_k = k

    return best_k

def transforming_conditional(s, num_components, mu_k, sig_k, p_k, dim_s):
    reg = 1e-6
    eig_floor = 1e-10
    s = np.asarray(s).reshape(-1)
    mu_cond = []
    cov_cond = []
    log_weights = []
    for k in range(num_components):
        mu = np.asarray(mu_k[k])
        sigma = np.asarray(sig_k[k]).copy()
        mu_s = mu[:dim_s]
        mu_xi = mu[dim_s:]
        sigma_ss = sigma[:dim_s, :dim_s].copy()
        sigma_sx = sigma[:dim_s, dim_s:].copy()
        sigma_xs = sigma[dim_s:, :dim_s].copy()
        sigma_xx = sigma[dim_s:, dim_s:].copy()
        sigma_ss_reg = sigma_ss + reg * np.eye(dim_s)
        try:
            sigma_ss_inv = np.linalg.inv(sigma_ss_reg)
        except np.linalg.LinAlgError:
            sigma_ss_inv = np.linalg.pinv(sigma_ss_reg)
        cond_mu = mu_xi + sigma_xs @ sigma_ss_inv @ (s - mu_s)
        cond_cov = sigma_xx - sigma_xs @ sigma_ss_inv @ sigma_sx
        cond_cov = 0.5 * (cond_cov + cond_cov.T)
        eigvals = np.linalg.eigvalsh(cond_cov)
        min_eig = eigvals.min()
        if min_eig < eig_floor:
            cond_cov += (eig_floor - min_eig + reg) * np.eye(cond_cov.shape[0])
        try:
            log_weight = (np.log(max(p_k[k], 1e-300))+multivariate_normal.logpdf(s, mean=mu_s, cov=sigma_ss_reg, allow_singular=True))
        except Exception:
            log_weight = -np.inf
        mu_cond.append(cond_mu)
        cov_cond.append(cond_cov)
        log_weights.append(log_weight)
    log_weights = np.asarray(log_weights)
    if not np.any(np.isfinite(log_weights)):
        weights = np.ones(num_components) / num_components
    else:
        max_log_weight = np.max(log_weights[np.isfinite(log_weights)])
        weights = np.exp(log_weights - max_log_weight)
        weights[~np.isfinite(weights)] = 0.0
        if weights.sum() <= 1e-12:
            weights = np.ones(num_components) / num_components
        else:
            weights = weights / weights.sum()

    return np.array(mu_cond), np.array(cov_cond), weights

def MC_sampling(K, N, mu_list, cov_list, p_list):
    d = mu_list.shape[1]
    samples = np.zeros((N, d))
    for i in range(N):
        k = np.random.choice(K, p=p_list)
        samples[i] = np.random.multivariate_normal(mu_list[k], cov_list[k])
    return samples

def oos_loss(q, s, h, b, W1, W2, dim_xi=1, seed=None):
    n = 100000
    rng = np.random.default_rng(seed)
    s = s.reshape(1, -1)
    eps1 = rng.uniform(-2, 2, size=(n, dim_xi))
    eps2 = rng.uniform(-2, 2, size=(n, dim_xi))
    lin  = s @ W1.T
    quad = (s**2) @ W2.T
    xi1 = lin + 50 + eps1
    xi2 = quad + 40 + eps2
    score = s[:, 0]
    prob = 0.5 * (1 + np.tanh(score))
    u = rng.random(n)
    xi_samples = np.where(u[:, None] < prob[:, None], xi1, xi2)
    q = np.array(q).reshape(1, -1)
    if q.shape[1] == 1 and dim_xi > 1:
        q = np.tile(q, (1, dim_xi))
    losses = h * np.maximum(q - xi_samples, 0) + b * np.maximum(xi_samples - q, 0)
    return np.mean(losses)

def oos_loss_valid(q, xi, h, b):
    loss = h * np.maximum(q - xi, 0) + b * np.maximum(xi - q, 0)
    return np.mean(loss)

def cv_GMM(N, max_K, eps_list, xi, s, h, b, hidden_node, hidden_layer, block_size, num_bins, total_epoch, device):
    dim_s, dim_xi = s.shape[1], xi.shape[1]
    latent_size = dim_s + dim_xi

    X = np.concatenate([s, xi], axis=1)
    split = int(len(X) * 0.9)
    train, val = X[:split], X[split:]

    s_train, xi_train = train[:, :dim_s], train[:, dim_s:]
    s_val, xi_val = val[:, :dim_s], val[:, dim_s:]

    scaler_s = StandardScaler().fit(s_train)
    scaler_xi = StandardScaler().fit(xi_train)

    s_train_std = scaler_s.transform(s_train)
    xi_train_std = scaler_xi.transform(xi_train)
    s_val_std = scaler_s.transform(s_val)

    data_train_std = np.concatenate([s_train_std, xi_train_std], axis=1)
    x_train_tensor = torch.tensor(data_train_std, dtype=torch.float32).to(device)

    cov_type = 'diag'
    K = fit_gmm_by_aic(data_train_std, max_K, covariance_type=cov_type, reg_covar=1e-2, n_init=10, random_state=42)
    base_gmm = GaussianMixture(n_components=K, covariance_type=cov_type, n_init=10, reg_covar=1e-2, random_state=42).fit(data_train_std)

    nfm, _, base_gmm = train_nf_model(latent_size=latent_size, best_K=K, 
                                      hidden_node=hidden_node, hidden_layer=hidden_layer, num_bins=num_bins, block_size=block_size, total_epoch=total_epoch,
                                      x=x_train_tensor, device=device, base_gmm=base_gmm, covariance_type=cov_type)

    s_std_list = []
    for j in range(s_val_std.shape[0]):
        s_j_std = s_val_std[j].ravel()
        s_std_list.append(np.hstack([s_j_std.reshape(1, -1), np.zeros((1,dim_xi))]))

    s_val_aug_std = np.vstack(s_std_list)
    s_tensor = torch.tensor(s_val_aug_std, dtype=torch.float32).to(device)

    z_s_val = inverse(nfm, s_tensor)[:, :dim_s]

    mu_base = base_gmm.means_
    cov_base = base_gmm.covariances_
    p_base = base_gmm.weights_
    sig_base = np.array([np.diag(cov) for cov in cov_base])

    num_mc_cv = 500
    xi_sampled_val_list = []

    for j in range(z_s_val.shape[0]):
        z_s_j = z_s_val[j].ravel()

        mu_cond, cov_cond, p_cond = transforming_conditional(s=z_s_j, num_components=K, mu_k=mu_base, sig_k=sig_base, p_k=p_base, dim_s=dim_s)
        z_xi_sample = MC_sampling(K, num_mc_cv, mu_cond, cov_cond, p_cond)
        z_full = np.hstack([np.repeat(z_s_j.reshape(1, -1), num_mc_cv, axis=0), z_xi_sample])

        z_tensor = torch.tensor(z_full, dtype=torch.float32).to(device)
        xi_sampled_std = forward(nfm, z_tensor)[:, dim_s:]
        xi_sampled = scaler_xi.inverse_transform(xi_sampled_std)
        xi_sampled = np.maximum(xi_sampled, 0)

        xi_sampled_val_list.append(xi_sampled)

    best_eps_result = {'eps': None, 'score': float('inf')}

    for eps in eps_list:
        valid_score = 0.0

        for j, xi_sampled in enumerate(xi_sampled_val_list):
            q_gmm = NewsVendor_2_Wass(xi_sampled, eps, h, b)
            valid_score += oos_loss_valid(q_gmm, xi_val[j], h, b)

        if valid_score < best_eps_result['score']:
            best_eps_result.update({'eps': eps, 'score': valid_score})

    return K, best_eps_result['eps']

def cv_GMM_nonNF(N, max_K, eps_list, xi, s, h, b, hidden_node, hidden_layer, block_size, num_bins, total_epoch, device):
    dim_s, dim_xi = s.shape[1], xi.shape[1]
    X = np.concatenate([s, xi], axis=1)           
    split = int(len(X) * 0.9)
    train, val = X[:split], X[split:]

    s_train, xi_train = train[:, :dim_s], train[:, dim_s:]
    s_val,   xi_val   = val[:,   :dim_s], val[:,   dim_s:]

    scaler_s  = StandardScaler().fit(s_train)
    scaler_xi = StandardScaler().fit(xi_train)

    s_train_std  = scaler_s.transform(s_train)
    xi_train_std = scaler_xi.transform(xi_train)
    s_val_std    = scaler_s.transform(s_val)      
    data_train_std = np.concatenate([s_train_std, xi_train_std], axis=1)

    cov_type = 'diag'
    K = fit_gmm_by_aic(data_train_std, max_K, covariance_type=cov_type, reg_covar=1e-2, n_init=10, random_state=42)
    gmm_x = GaussianMixture(n_components=K, covariance_type=cov_type, n_init=10, reg_covar=1e-2).fit(data_train_std)
    mu_x, cov_x, p_x = gmm_x.means_, gmm_x.covariances_, gmm_x.weights_
    sig_x = np.array([np.diag(cov) for cov in cov_x]) if cov_type == 'diag' else cov_x
    
    best_eps_result = {'eps': None, 'score': float('inf')}
    num_mc = 500

    for eps in eps_list:
        valid_score = 0.0

        for j in range(s_val.shape[0]):
            s_j = s_val_std[j].ravel()  
            mu_cond_x, cov_cond_x, p_cond_x = transforming_conditional(s=s_j, num_components=K,mu_k=mu_x, sig_k=sig_x, p_k=p_x, dim_s=dim_s)
            xi_sampled_std = MC_sampling(K, num_mc, mu_cond_x, cov_cond_x, p_cond_x)
            xi_sampled = scaler_xi.inverse_transform(xi_sampled_std)
            xi_sampled = np.maximum(xi_sampled, 0)
            q_gmm= NewsVendor_2_Wass(xi_sampled, eps, h, b)
            valid_score += oos_loss_valid(q_gmm, xi_val[j], h, b)

        if valid_score < best_eps_result['score']:
            best_eps_result.update({'eps': eps, 'score': valid_score})

    return K, best_eps_result['eps']

In [ ]:
############################### RNW function #########################
def NW_weights(s_test, s_is, H):
    N = len(s_is)
    numerator = np.zeros(N)

    H = float(H)
    if not np.isfinite(H) or H <= 0:
        return np.ones(N) / N

    s_is_arr = np.asarray(s_is, dtype=float)
    if s_is_arr.ndim == 1:
        s_is_arr = s_is_arr.reshape(-1, 1)
    s_test_arr = np.asarray(s_test, dtype=float).reshape(-1)

    S = np.cov(s_is_arr.T, ddof=1)
    S = np.asarray(S, dtype=float)
    if S.ndim == 0:
        S = np.asarray([[float(S)]])
    S = S + 1e-8 * np.eye(S.shape[0])

    for i in range(N):
        diff = s_is_arr[i] - s_test_arr
        sol = np.linalg.solve(S, diff)
        quad = max(float(diff @ sol), 0.0)
        numerator[i] = np.exp(-0.5 * np.sqrt(quad / max(H, 1e-12)))

    denominator = numerator.sum()
    weight = numerator / denominator if denominator != 0 else np.ones(N) / N
    weight = weight / weight.sum() if weight.sum() != 0 else np.ones(N) / N
    return weight
def cv_lda(Cs, C_H, xi_is, s_is, h, b):
    valid_scores = []

    split = int(len(xi_is) * 0.9)
    xi_subtrain, xi_valid = xi_is[:split], xi_is[split:]
    s_subtrain, s_valid = s_is[:split], s_is[split:]
    N = xi_subtrain.shape[0]
    dim_s = s_subtrain.shape[1]
    H = C_H * 1 / (N ** (1 / 6))

    use_kde = dim_s <= 5 

    if use_kde:
        kde = KernelDensity(kernel='exponential', bandwidth=1).fit(s_subtrain)

    for C in Cs:
        valid_score = 0
        for j in range(len(s_valid)):
            s_val = s_valid[j]
            if use_kde:
                g_s = np.exp(kde.score_samples(s_val.reshape(1, -1))).item()
            else:
                g_s = 1.0  

            lda_0 = 1 / np.sqrt(N * (H ** 2) * g_s)
            weight = NW_weights(s_val, s_subtrain, H)
            params = {
                "b": b,
                "h": h,
                "s_test": s_val,
                "xi_is": xi_subtrain,
                "weight": weight,
            }
            lda = lda_0 * C
            News = Newsvendor(reg=lda, verbose=False)
            News.fit(params)
            q = News.coef_
            valid_score += oos_loss_valid(q, xi_valid[j], h, b)

        valid_score /= len(s_valid)
        valid_scores.append(valid_score)

    idx = np.argmin(valid_scores)
    return Cs[idx]

def cv_lda2(Cs,C_H,xi_is,s_is,H,h,b):
    valid_scores=[]

    split = int(len(xi_is) * 0.9)
    xi_subtrain, xi_valid = xi_is[:split], xi_is[split:]
    s_subtrain, s_valid = s_is[:split], s_is[split:]
    
    N = xi_subtrain.shape[0]
    H=C_H * 1/(N**(1/6))
    lda_0 = 1/np.sqrt(N*(H**2))

    for i,C in enumerate(Cs):
        valid_score = 0
        for j in range(s_valid.shape[0]):
            s_val = s_valid[j]
            weight=NW_weights(s_val,s_subtrain,H)
            params={
            "b":b,
            "h":h,
            "s_test": s_val,
            "xi_is":xi_subtrain,
            "weight":weight,
            }
            lda = C * lda_0
            News=Newsvendor(reg=lda,verbose=False)
            News.fit(params)
            q=News.coef_
            valid_score += oos_loss_valid(q,np.array([xi_valid[j]]),h,b)
        valid_scores.append(valid_score)
    idx=np.argmin(valid_scores)
    return(Cs[idx])

def cv_H(Cs, xi_is, s_is, h, b):
    valid_scores = []

    split = int(len(xi_is) * 0.9)
    xi_subtrain, xi_valid = xi_is[:split], xi_is[split:]
    s_subtrain, s_valid = s_is[:split], s_is[split:]
    N = xi_subtrain.shape[0]
    H_0 = 1 / (N ** (1 / 6))

    for i, C in enumerate(Cs):
        valid_score = 0
        H = C * H_0

        for j in range(s_valid.shape[0]):
            s_val = s_valid[j]
            weight = NW_weights(s_val, s_subtrain, H)

            weight = np.round(weight, 6)
            weight = weight / weight.sum()
            params = {
                "b": b,
                "h": h,
                "s_test": s_val,
                "xi_is": xi_subtrain,
                "weight": weight,
            }

            News = Newsvendor(reg=0, verbose=False)
            News.fit(params)
            q = News.coef_
            valid_score += oos_loss_valid(q, xi_valid[j], h, b)

        valid_scores.append(valid_score)

    idx = np.argmin(valid_scores)
    return Cs[idx]

In [ ]:
########################### ResDRO function ##############################
def NewsVendor_1_Wass(xi_is, eps, h, b):
    scale = 100
    xi_is = xi_is.astype(float) / scale
    eps = eps / scale

    N = xi_is.shape[0]

    lda = cp.Variable(nonneg=True)
    s = cp.Variable(N)
    z = cp.Variable((N, 2))
    q = cp.Variable(nonneg=True)

    const = []

    for i in range(N):
        const.append(h * q + z[i,0] * xi_is[i] <= s[i])
        const.append(-b * q + z[i,1] * xi_is[i] <= s[i])
        const.append(z[i,0] >= -h)
        const.append(z[i,1] >= b)
        for k in range(2):
            const.append(cp.norm_inf(z[i,k]) <= lda)

    obj = cp.Minimize(lda * eps + (1 / N) * cp.sum(s))
    prob = cp.Problem(obj, const)
    prob.solve(solver=cp.MOSEK)

    return q.value * scale

def cv_eps_ResDRO(eps_list, xi_is, s_is, h, b, val_size=10):
    xi_is = np.asarray(xi_is)
    s_is = np.asarray(s_is)

    n = len(xi_is)
    if n < 2:
        return 0.0
    
    n_valid = min(val_size, max(1, n // 5), n - 1)
    split = n - n_valid

    xi_subtrain, xi_valid = xi_is[:split], xi_is[split:]
    s_subtrain, s_valid = s_is[:split], s_is[split:]

    model = LinearRegression().fit(s_subtrain, xi_subtrain)
    residuals = xi_subtrain - model.predict(s_subtrain)

    valid_scores = []

    for eps in eps_list:
        valid_score = 0.0

        for j in range(len(s_valid)):
            s_j = s_valid[j]
            xi_j = xi_valid[j]

            f_hat = model.predict(s_j.reshape(1, -1)).item()
            xi_ER = np.maximum(residuals + f_hat, 0)

            q = NewsVendor_1_Wass(xi_ER, eps, h, b)
            valid_score += oos_loss_valid(q, xi_j, h, b)

        valid_scores.append((eps, valid_score))

    best_eps, _ = min(valid_scores, key=lambda x: x[1])
    return best_eps

In [ ]:
############################### LDR function ###############################
def LDR(xi_is, s_is, h, b, verbose=False):
    xi_is = np.asarray(xi_is, dtype=float)
    s_is = np.asarray(s_is, dtype=float)
    N, dim_xi = xi_is.shape
    _, dim_s = s_is.shape

    beta0 = cp.Variable(dim_xi)
    B = cp.Variable((dim_s, dim_xi))

    xi_hat = s_is @ B + beta0

    h_vec = np.asarray(h, dtype=float)
    b_vec = np.asarray(b, dtype=float)

    excess = cp.Variable((N, dim_xi), nonneg=True)
    shortage = cp.Variable((N, dim_xi), nonneg=True)

    constraints = [excess >= xi_hat - xi_is, shortage >= xi_is - xi_hat]

    loss = cp.sum(cp.multiply(h_vec.reshape(1, -1), excess) + cp.multiply(b_vec.reshape(1, -1), shortage)) / N

    prob = cp.Problem(cp.Minimize(loss), constraints)
    prob.solve(solver=cp.MOSEK, verbose=verbose)

    if beta0.value is None or B.value is None:
        raise RuntimeError(f"LDR failed. Status: {prob.status}")

    rule = {
        "beta0": np.asarray(beta0.value).reshape(-1),
        "B": np.asarray(B.value),
        "objective": prob.value,
        "status": prob.status,
    }

    return rule

def LDR_decision(s_test, rule):
    s_test = np.asarray(s_test, dtype=float)
    beta0 = rule["beta0"]
    B = rule["B"]

    x = s_test @ B + beta0
    x = x.reshape(-1)

    x = np.maximum(x, 0)

    return x

In [ ]:
############################### NN-DR function ###############################
def decision_loss_fn(q_pred, xi_true, h_tensor, b_tensor):
    excess = torch.relu(q_pred - xi_true)
    shortage = torch.relu(xi_true - q_pred)

    loss = h_tensor * excess + b_tensor * shortage

    return loss.sum(dim=1).mean()

def NN(xi_is, s_is, h, b, hidden_node, hidden_layer, lr, weight_decay, total_epoch, batch_size, val_rate, patience, min_delta, 
       seed=None, device="cpu", verbose=False):
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    xi_is = np.asarray(xi_is, dtype=float)
    s_is = np.asarray(s_is, dtype=float)
    N, dim_xi = xi_is.shape
    N_s, dim_s = s_is.shape

    h_vec = np.asarray(h, dtype=float)
    b_vec = np.asarray(b, dtype=float)

    rng = np.random.default_rng(seed)
    indices = np.arange(N)
    rng.shuffle(indices)

    N_val = int(val_rate * N)
    N_val = max(1, min(N_val, N - 1))

    val_idx = indices[:N_val]
    train_idx = indices[N_val:]

    s_train = s_is[train_idx]
    xi_train = xi_is[train_idx]

    s_val = s_is[val_idx]
    xi_val = xi_is[val_idx]

    s_mean = np.zeros((1, dim_s))
    s_std = np.ones((1, dim_s))

    s_train_fit = s_train
    s_val_fit = s_val

    X_train = torch.tensor(s_train_fit, dtype=torch.float32).to(device)
    Y_train = torch.tensor(xi_train, dtype=torch.float32).to(device)

    X_val = torch.tensor(s_val_fit, dtype=torch.float32).to(device)
    Y_val = torch.tensor(xi_val, dtype=torch.float32).to(device)

    h_tensor = torch.tensor(h_vec.reshape(1, -1), dtype=torch.float32).to(device)
    b_tensor = torch.tensor(b_vec.reshape(1, -1), dtype=torch.float32).to(device)

    N_train = X_train.shape[0]

    if batch_size is None:
        batch_size = N_train

    layers = []
    input_dim = dim_s

    for _ in range(hidden_layer):
        layers.append(nn.Linear(input_dim, hidden_node))
        layers.append(nn.ReLU())
        input_dim = hidden_node

    layers.append(nn.Linear(input_dim, dim_xi))
    layers.append(nn.Softplus())

    model = nn.Sequential(*layers).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay,)

    best_val_loss = np.inf
    best_train_loss = np.inf
    best_state = None
    patience_counter = 0
    best_epoch = 0

    for epoch in range(total_epoch):
        model.train()

        perm = torch.randperm(N_train, device=device)
        train_loss_epoch = 0.0

        for start in range(0, N_train, batch_size):
            idx = perm[start:start + batch_size]

            x_batch = X_train[idx]
            y_batch = Y_train[idx]

            q_pred = model(x_batch)
            loss = decision_loss_fn(q_pred, y_batch, h_tensor, b_tensor)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_epoch += loss.item() * len(idx)

        train_loss_epoch /= N_train

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = decision_loss_fn(val_pred, Y_val, h_tensor, b_tensor).item()

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_train_loss = train_loss_epoch
            best_epoch = epoch + 1
            patience_counter = 0

            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()

    rule = {
        "model": model,
        "s_mean": s_mean,
        "s_std": s_std,
        "dim_s": dim_s,
        "dim_xi": dim_xi,
        "best_train_loss": best_train_loss,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "device": device,
        "loss_type": "newsvendor_decision_loss",
    }

    return rule

def NN_decision(s_test, rule):
    s_test = np.asarray(s_test, dtype=float)

    model = rule["model"]
    s_mean = rule["s_mean"]
    s_std = rule["s_std"]
    device = rule["device"]

    s_fit = (s_test - s_mean) / s_std
    s_test_tensor = torch.tensor(s_fit, dtype=torch.float32,).to(device)
    model.eval()

    with torch.no_grad():
        x = model(s_test_tensor).cpu().numpy()

    return x.reshape(-1)

In [ ]:
# ----------------------- Main ----------------------------
T = 50
dim_s = 1
dim_xi = 1
N_list = [50, 100, 200, 400]
a_list = [1, 5, 9]  
b_list = [-2, -1, 0, 1]
num_mc = 300
Cs = CHs = [1, 5, 10]
h, b =  10, 2
max_K = 3 
hidden_node, hidden_layer, block_size, bins, total_epoch = 32, 1, 1, 8, 500
device = "cpu"
eps_list = make_eps_grid(a_list, b_list)

no_cv_eps_N = {N for N in N_list if N == 400}   

all_results = []

for N in N_list:
    print(f"Running trials for dim_s = {dim_s}, N = {N}")

    def run_trial(tt, N):
        max_retry = 100

        for attempt_trial in range(max_retry):
            seed = 1000 * dim_s + 10 * tt + attempt_trial
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            data_rng = np.random.default_rng(seed)

            seed_train = int(data_rng.integers(1e9))
            seed_test  = int(data_rng.integers(1e9))
            seed_oos   = int(data_rng.integers(1e9))

            try:
                trial_start = perf_counter()
                W1 = 0.3 * np.ones((dim_xi, dim_s))
                W2 = 5 * np.ones((dim_xi, dim_s))

                s_is, xi_is = generate_data(N, dim_s, dim_xi, W1, W2, seed=seed_train)
                s_test, _ = generate_data(1, dim_s, dim_xi, W1, W2, seed=seed_test)
                s_test = s_test[0]

                # ------------------------------- LDR -------------------------------
                t0 = perf_counter()
                ldr_rule = LDR(xi_is=xi_is, s_is=s_is, h=h, b=b, verbose=False)
                x_LDR = LDR_decision(s_test=s_test, rule=ldr_rule)
                loss_LDR = oos_loss(x_LDR, s_test, h, b, W1, W2, dim_xi=dim_xi, seed=seed_oos)
                time_LDR = perf_counter() - t0
                print(f"LDR Finished {tt} trial for N = {N}, loss={loss_LDR}, time={time_LDR:.3f}s")

                # ------------------------------- NN-DR -------------------------------
                t0 = perf_counter()
                NN_DR = NN(xi_is=xi_is, s_is=s_is, h=h, b=b,
                           hidden_node=32, hidden_layer=1, lr=1e-3, weight_decay=1e-2, total_epoch=total_epoch, batch_size=64, val_rate=0.1, patience=10, min_delta=1e-4, 
                           seed=seed, device=device, verbose=False)
                x_NN_DR = NN_decision(s_test=s_test, rule=NN_DR)
                loss_NN_DR = oos_loss(x_NN_DR, s_test, h, b, W1, W2, dim_xi=dim_xi, seed=seed_oos)
                time_NN_DR = perf_counter() - t0
                print(f"NN-DR Finished {tt} trial for N = {N}, loss={loss_NN_DR}, time={time_NN_DR:.3f}s")

                # -------------------------------- RNW --------------------------------
                t0 = perf_counter()
                if N in no_cv_eps_N:
                    C_H = 1.0
                    H = C_H * 1 / (N ** (1 / 6))
                    kde = KernelDensity(kernel='exponential', bandwidth=1).fit(s_is)
                    g_s = np.exp(kde.score_samples(s_test.reshape(1, -1))).item()
                    weight = NW_weights(s_test, s_is, H)
                    C_smart = 0.0
                    lda_smart = 0.0
                else:
                    C_H = cv_H(CHs, xi_is, s_is, h, b)
                    H = C_H * 1 / (N ** (1 / 6))
                    kde = KernelDensity(kernel='exponential', bandwidth=1).fit(s_is)
                    g_s = np.exp(kde.score_samples(s_test.reshape(1, -1))).item()
                    weight = NW_weights(s_test, s_is, H)
                    C_smart = cv_lda(Cs, C_H, xi_is, s_is, h, b)
                    lda_0 = 1 / np.sqrt(N * (H ** 2) * g_s)
                    lda_smart = C_smart * lda_0

                RNW_model = Newsvendor(reg=lda_smart, verbose=False)
                RNW_model.fit({"b": b, "h": h, "s_test": s_test, "xi_is": xi_is, "weight": weight})
                q_RNW = RNW_model.coef_
                loss_RNW = oos_loss(q_RNW, s_test, h, b, W1, W2, dim_xi=dim_xi, seed=seed_oos)
                time_RNW = perf_counter() - t0
                print(f"RNW Finished {tt} trial for N = {N}, loss={loss_RNW}, time={time_RNW:.3f}s")

                # --------------------------- NF-GMM -------------------------------
                t0 = perf_counter()
                if N in no_cv_eps_N:
                    best_eps_NF = 0.0
                else:
                    best_K_nf, best_eps_NF = cv_GMM(N, max_K, eps_list, xi_is, s_is, h, b,
                                            hidden_node, hidden_layer, block_size, bins, total_epoch, device)
                scaler_s, scaler_xi = StandardScaler(), StandardScaler()
                s_std = scaler_s.fit_transform(s_is)
                xi_std = scaler_xi.fit_transform(xi_is)

                s_val = s_test.reshape(1, -1)
                s_val_std = scaler_s.transform(s_val)

                data_std = np.hstack([s_std, xi_std])
                x_tensor = torch.tensor(data_std, dtype=torch.float32).to(device)

                cov_type_NF = 'diag'
                if N in no_cv_eps_N: 
                    best_K_nf = fit_gmm_by_aic(data_std, max_K=max_K, covariance_type=cov_type_NF, reg_covar=1e-2, n_init=10, random_state=42)
                else: None 
                base_gmm = GaussianMixture(n_components=best_K_nf, covariance_type=cov_type_NF, n_init=10, reg_covar=1e-2, random_state=42).fit(data_std)

                nfm, _, base_gmm = train_nf_model(latent_size=dim_s + dim_xi, best_K=best_K_nf,
                                                  hidden_node=hidden_node, hidden_layer=hidden_layer, block_size=block_size, num_bins=bins, total_epoch=total_epoch, x=x_tensor, device=device, base_gmm=base_gmm, covariance_type=cov_type_NF)
                s_test_std = np.hstack([s_val_std, np.zeros((1, dim_xi))])
                s_test_tensor = torch.tensor(s_test_std, dtype=torch.float32).to(device)

                z_s = inverse(nfm, s_test_tensor)[:, :dim_s].reshape(-1)

                mu_base = base_gmm.means_
                cov_base = base_gmm.covariances_
                p_base = base_gmm.weights_
                sig_base = np.array([np.diag(cov) for cov in cov_base])
                mu_cond_z, cov_cond_z, w_z = transforming_conditional(s=z_s, num_components=best_K_nf, mu_k=mu_base, sig_k=sig_base, p_k=p_base, dim_s=dim_s)

                z_xi_sample = MC_sampling(best_K_nf, num_mc, mu_cond_z, cov_cond_z, w_z)
                z_full = np.hstack([np.repeat(z_s.reshape(1, -1), len(z_xi_sample), axis=0),z_xi_sample])

                z_tensor = torch.tensor(z_full, dtype=torch.float32).to(device)
                xi_sampled_std = forward(nfm, z_tensor)[:, dim_s:]
                xi_sampled = scaler_xi.inverse_transform(xi_sampled_std)
                xi_sampled = np.maximum(xi_sampled, 0)

                q_NF_gmm = NewsVendor_2_Wass(xi_sampled, best_eps_NF, h, b)
                loss_NF_gmm = oos_loss(q_NF_gmm, s_test, h, b, W1, W2, dim_xi=dim_xi, seed=seed_oos)
                time_NF_gmm = perf_counter() - t0
                print(f"NF-GMM Finished {tt} trial for N = {N}, loss={loss_NF_gmm}, time={time_NF_gmm:.3f}s")

                # -------------------- Non-NF-GMM --------------------------
                t0 = perf_counter()
                if N in no_cv_eps_N:
                    best_eps_nonNF = 0.0
                else:
                    best_K_nonNF, best_eps_nonNF = cv_GMM_nonNF(N, max_K, eps_list, xi_is, s_is, h, b,
                                                                hidden_node, hidden_layer, block_size, bins, total_epoch, device)
                scaler_xi, scaler_s = StandardScaler(), StandardScaler()
                s_std = scaler_s.fit_transform(s_is)
                xi_std = scaler_xi.fit_transform(xi_is)
                data_std_nonNF = np.hstack([s_std, xi_std])

                cov_type_nonNF = 'diag'
                if N in no_cv_eps_N: 
                    best_K_nonNF = fit_gmm_by_aic(data_std_nonNF, max_K, cov_type_nonNF, reg_covar=1e-2, n_init=10, random_state=42)
                else: None 
                gmm_x = GaussianMixture(n_components=best_K_nonNF, covariance_type=cov_type_nonNF, n_init=10, reg_covar=1e-2).fit(data_std_nonNF)
                mu_x, cov_x, p_x = gmm_x.means_, gmm_x.covariances_, gmm_x.weights_
                sig_x = np.array([np.diag(cov) for cov in cov_x]) if cov_type_nonNF == 'diag' else cov_x

                s_val = s_test.reshape(1, -1)
                s_val_std = scaler_s.transform(s_val)
                s_vec = s_val_std.ravel()
                mu_cond_x_nonNF, cov_cond_x_nonNF, p_cond_x_nonNF = transforming_conditional(s=s_vec, num_components=best_K_nonNF, mu_k=mu_x, sig_k=sig_x, p_k=p_x, dim_s=dim_s)

                xi_sampled_std = MC_sampling(best_K_nonNF, num_mc, mu_cond_x_nonNF, cov_cond_x_nonNF, p_cond_x_nonNF)
                xi_sampled = scaler_xi.inverse_transform(xi_sampled_std)
                xi_sampled = np.maximum(xi_sampled, 0)

                q_nonNF_gmm = NewsVendor_2_Wass(xi_sampled, best_eps_nonNF, h, b)
                loss_nonNF_gmm = oos_loss(q_nonNF_gmm, s_test, h, b, W1, W2, dim_xi=dim_xi, seed=seed_oos)
                time_nonNF_gmm = perf_counter() - t0
                print(f"Non-NF-GMM Finished {tt} trial for N = {N}, loss={loss_nonNF_gmm}, time={time_nonNF_gmm:.3f}s")

                # --------------------------- ResDRO -----------------------------
                t0 = perf_counter()
                if N in no_cv_eps_N:
                    best_eps2 = 0.0
                else:
                    best_eps2 = cv_eps_ResDRO(eps_list, xi_is, s_is, h, b)
                model = LinearRegression().fit(s_is, xi_is)
                residuals = xi_is - model.predict(s_is)
                f_hat = model.predict(np.atleast_2d(s_test)).item()
                xi_er = np.maximum(residuals + f_hat, 0)

                q_ResDRO = NewsVendor_1_Wass(xi_er, best_eps2, h=h, b=b)
                loss_ResDRO = oos_loss(q_ResDRO, s_test, h, b, W1, W2, dim_xi=dim_xi, seed=seed_oos)    

                time_ResDRO = perf_counter() - t0
                print(f"ResDRO Finished {tt} trial for N = {N}, loss={loss_ResDRO}, time={time_ResDRO:.3f}s")

                time_total = perf_counter() - trial_start
                print(f"NF-gmm, nonNF-gmm, ResDRO, RNW, LDR, NN-DR losses: "f"{loss_NF_gmm}, {loss_nonNF_gmm}, {loss_ResDRO}, {loss_RNW}, {loss_LDR}, {loss_NN_DR}")

                return {
                    'K_NF': best_K_nf,
                    'K_nonNF_GMM': best_K_nonNF,
                    'eps_NF_GMM': best_eps_NF,
                    'eps_nonNF_GMM': best_eps_nonNF,
                    'eps_ResDRO': best_eps2,

                    'loss_NF_GMM': loss_NF_gmm,
                    'loss_nonNF_GMM': loss_nonNF_gmm,
                    'loss_ResDRO': loss_ResDRO,
                    'loss_RNW': loss_RNW,
                    'loss_LDR': loss_LDR,
                    'loss_NN_DR': loss_NN_DR,

                    'time_NF_gmm': time_NF_gmm,
                    'time_nonNF_gmm': time_nonNF_gmm,
                    'time_ResDRO': time_ResDRO,
                    'time_RNW': time_RNW,
                    'time_LDR': time_LDR,
                    'time_NN_DR': time_NN_DR,
                    'time_total': time_total
                }

            except Exception as e:
                print(f"[Global Retry {attempt_trial + 1}/{max_retry}] Trial {tt}, N={N} failed: {e}")
                continue

        return None

    results = Parallel(n_jobs=-1)(delayed(run_trial)(tt, N) for tt in tqdm(range(T)))

    results = [r for r in results if isinstance(r, dict) and r is not None]

    if len(results) == 0:
        print(f"⚠️ No valid results for N={N}, skipping.")
        continue

    for i, r in enumerate(results):
        r["Trial"] = i
        r["N"] = N

    df = pd.DataFrame(results)

    cols = ["N", "Trial"] + [c for c in df.columns if c not in ["N", "Trial"]]
    df = df[cols]

    mean_row = {"N": N, "Trial": "AVG"}

    for col in df.columns:
        if col not in ["N", "Trial"] and pd.api.types.is_numeric_dtype(df[col]):
            mean_row[col] = df[col].mean(skipna=True)

    df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

    out_dir = Path("Results")
    out_dir.mkdir(parents=True, exist_ok=True)

    save_path = out_dir / f"NV_{dim_s}d_LinQuad_N{N}.csv"
    df.to_csv(save_path, index=False)

    print("✅ Saved NV results to:", save_path)
    print(f"✅ AVG row includes average computational times for N={N}")
    
    all_results.append(df)